Data Preprocessing:
1.Load the dataset into a suitable data structure (e.g., pandas DataFrame).
2.Handle missing values, if any.
3.Explore the dataset to understand its structure and attributes.


In [3]:
import pandas as pd

df = pd.read_csv("anime.csv")
df.head()


,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


In [6]:
# Check missing
df.isnull().sum()



anime_id     0
name         0
genre        0
type        25
episodes     0
rating       0
members      0
dtype: int64

In [ ]:
Feature Extraction:

Decide on the features that will be used for computing similarity (e.g., genres, user ratings).
Convert categorical features into numerical representations if necessary.
Normalize numerical features if required


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Clean data
df['genre'] = df['genre'].fillna('')
df['genre'] = df['genre'].astype(str)

# TF-IDF
tfidf = TfidfVectorizer(stop_words='english')

genre_matrix = tfidf.fit_transform(df['genre'])

# Output check
print("Shape:", genre_matrix.shape)
print("Features:", tfidf.get_feature_names_out()[:10])

Shape: (12294, 46)
Features: ['action' 'adventure' 'ai' 'arts' 'cars' 'comedy' 'dementia' 'demons'
 'drama' 'ecchi']


In [12]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

# Fix data
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')

df['rating'] = df['rating'].fillna(df['rating'].mean())
df['episodes'] = df['episodes'].fillna(df['episodes'].median())
df['members'] = df['members'].fillna(df['members'].median())

# Scaling
scaler = MinMaxScaler()
num_features = df[['rating', 'episodes', 'members']]
num_scaled = scaler.fit_transform(num_features)

# Output
num_scaled_df = pd.DataFrame(num_scaled, columns=['rating', 'episodes', 'members'])
print(num_scaled_df.head())

     rating  episodes   members
0  0.924370  0.000000  0.197872
1  0.911164  0.034673  0.782770
2  0.909964  0.027518  0.112689
3  0.900360  0.012658  0.664325
4  0.899160  0.027518  0.149186


In [13]:
import numpy as np
from scipy.sparse import hstack, csr_matrix

# Convert numeric to sparse
num_scaled_sparse = csr_matrix(num_scaled)

# Combine
final_features = hstack([genre_matrix, num_scaled_sparse])

# Output check
print("Shape:", final_features.shape)
print("Sample:", final_features.toarray()[0][:10])

Shape: (12294, 49)
Sample: [0.         0.         0.         0.         0.         0.
 0.         0.         0.44024715 0.        ]


Recommendation System:

Design a function to recommend anime based on cosine similarity.
Given a target anime, recommend a list of similar anime based on cosine similarity scores.
Experiment with different threshold values for similarity scores to adjust the recommendation list size.
Analyze the performance of the recommendation system and identify areas of improvement.


In [38]:

import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity

# -----------------------------
# Sample Dataset
# -----------------------------
data = {
    'name': ["Kimi no Na wa.", "Fullmetal Alchemist: Brotherhood", "Gintama°", "Steins;Gate", "Gintama'"],
    'genre': ["Drama, Romance, School, Supernatural",
              "Action, Adventure, Drama, Fantasy, Magic, Military",
              "Action, Comedy, Historical, Parody, Samurai, Shounen",
              "Sci-Fi, Thriller",
              "Action, Comedy, Historical, Parody, Samurai, Shounen"]
}
df = pd.DataFrame(data)
# Preprocess Genre
df['genre_list'] = df['genre'].apply(lambda x: x.split(', '))
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(df['genre_list'])
# Cosine Similarity Calculation
similarity_matrix = cosine_similarity(genre_matrix)

# Recommendation Function
def recommend_anime_cosine(anime_name, threshold=0.3, top_n=5):
    if anime_name not in df['name'].values:
        return "Anime not found"
    
    idx = df[df['name'] == anime_name].index[0]
    scores = list(enumerate(similarity_matrix[idx]))
    # Sort by similarity descending
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    
    recommendations = []
    for i, score in scores:
        if i != idx and score >= threshold:  # apply threshold
            recommendations.append((df.iloc[i]['name'], round(score, 2)))
    
    return recommendations[:top_n]

# -----------------------------
# Example Run
# -----------------------------
print("=== Cosine Similarity Recommendations ===")
print(recommend_anime_cosine("Kimi no Na wa.", threshold=0.3))

=== Cosine Similarity Recommendations ===
[]


1. What is a Recommendation System?
A recommendation system is a technique used to suggest items (here, anime) to users based on their preferences or similarity between items.
For example, if a user likes action anime, the system will recommend other similar action anime.

Performance Analysis & Improvements
Observations:
Genre dominates recommendations because it’s high-dimensional.
Rating and members have less influence (can adjust weight by multiplying features).
Improvements:
Weight numerical features higher if you want ratings or episodes to matter more.
Include content-based features like studio, type (TV/OVA), etc.
Use collaborative filtering for better personalization.

What is Threshold?
Threshold is the minimum similarity score required to include an anime in the recommendation list.
Example:
	•	Threshold = 0.5 → More recommendations (less strict)
	•	Threshold = 0.8 → Fewer but highly similar recommendations

Experimenting with Threshold Values

Different threshold values affect the number and quality of recommendations:

Threshold        Result
                
0.4              Large number of recommendations
0.6              Balanced results
0.8              Highly accurate but fewer recommendations
    


Performance Analysis

Advantages:
	•	Simple and easy to implement
	•	Works well for content-based filtering
	•	Fast computation

Limitations:
	•	Does not consider user preferences
	•	Cold start problem for new anime
	•	Limited to available features

Areas of Improvement

The recommendation system can be improved by:
	•	Using Collaborative Filtering
	•	Including user ratings and reviews
	•	Building a Hybrid Recommendation System
	•	Applying Machine Learning or Deep Learning models

final conclsion :
A cosine similarity-based recommendation system is simple and effective for recommending similar anime.
However, it can be enhanced by incorporating user behavior and advanced techniques to improve accuracy and personalization

In [ ]:
Interview Questions:
1. Can you explain the difference between user-based and item-based collaborative filtering?
2. What is collaborative filtering, and how does it work?

1.-->
| Aspect          | **User-Based**                                                                                                        | **Item-Based**                                                                                            |
| --------------- | --------------------------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------- |
| **Definition**  | Finds users similar to the target user and recommends items they liked.                                               | Finds items similar to the item the user liked and recommends them.                                       |
| **Focus**       | Similarity between users.                                                                                             | Similarity between items.                                                                                 |
| **Example**     | User A likes Naruto and One Piece. User B likes Naruto. Recommend One Piece to User B because similar users liked it. | User likes Naruto. Recommend other anime like Naruto (e.g., Bleach, Fairy Tail) based on item similarity. |
| **Scalability** | Harder for large user bases.                                                                                          | More scalable because number of items < number of users in most cases.                                    |


 Key idea:

User-based = “People like you liked this.”
Item-based = “Because you liked this item, you may like similar items.”


2.-->Definition:
Collaborative filtering is a recommendation technique that predicts a user’s interest in items based on the behavior and preferences of many users.

How it works:

Collect user-item interaction data – e.g., ratings, views, clicks.
Compute similarities:
User-based: find similar users based on rating patterns.
Item-based: find similar items based on how users rated them.
Generate predictions:
For a user, predict the rating of an item using similar users’ ratings (user-based) or similar items’ ratings (item-based).
Recommend:
Recommend items with the highest predicted ratings.

Example:

Netflix sees that you liked Stranger Things. It finds other users with similar tastes or finds similar shows and recommends The Umbrella Academy.
    

